# 02. LLM 전문가 리뷰 (Option A)

## 이 노트북이 하는 일
1. Rule 매핑 결과를 **LLM(GPT-4.1)이 리뷰**
2. **점수는 바꾸지 않음** — LLM 의견(score, feedback)만 메타데이터로 기록
3. V1 Rule 데이터 + LLM 리뷰 → `project_features_option_a.parquet` 생성

## 왜 Option A?
LLM이 점수를 바꾸면 **재현성 상실** + **NLP 논문화**.  
→ LLM = 전문가 리뷰어, 점수 = Rule 고정.

## 주의
- 이 노트북은 **OpenAI API 비용 발생** (75건 = $9)
- 비용 관리 위해 기본적으로 75건만 샘플 리뷰
- 전수(460건) 원하면 `MAX_SAMPLES = None` 로 변경

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass

import os
import pandas as pd
from src.langgraph_workflow.graph import run_standardization
from src.data.loader import LEEDDataLoader

print(f'OPENAI_API_KEY: {"✓" if os.environ.get("OPENAI_API_KEY") else "✗ 설정 필요"}')

## 설정

In [ ]:
MAX_SAMPLES = 75  # None으로 하면 전수 (460건, $55 예상)
SCORECARD_DIR = ROOT / 'data' / 'raw' / 'scorecards'
OUTPUT_DIR = ROOT / 'results' / 'llm_review'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1단계: LLM 리뷰 실행

LangGraph 파이프라인(V2)이 각 건물에 대해:
1. Rule로 v5 점수 계산
2. 수학 검증
3. **LLM이 의미적 타당성 검증** (target="rule")
4. 결과 저장 (점수는 Rule 유지, LLM 판단은 메타)

In [ ]:
loader = LEEDDataLoader()
csv_df = loader.load_project_directory()

pdf_files = sorted(SCORECARD_DIR.glob('*.pdf'))
if MAX_SAMPLES:
    pdf_files = pdf_files[:MAX_SAMPLES]

print(f'처리 대상: {len(pdf_files)}개 PDF')

In [ ]:
log_rows = []
for i, pdf in enumerate(pdf_files, 1):
    if i % 5 == 0 or i == 1 or i == len(pdf_files):
        print(f'[{i:3d}/{len(pdf_files)}] {pdf.name[:55]}')
    try:
        state = run_standardization(pdf_path=str(pdf), directory_df=csv_df)
        if state.get('status') != 'completed':
            continue
        
        final = state['final_v5_data']
        val = state.get('validation_result', {}) or {}
        math_r = state.get('math_validation_result', {}) or {}
        
        log_rows.append({
            'pdf': pdf.name,
            'project_id': final.get('project_id', ''),
            'version': final.get('original_version', ''),
            'certification_level': final.get('certification_level', ''),
            'drift': math_r.get('ratio_drift', 0),
            'val_target': val.get('target'),
            'val_is_valid': val.get('is_valid'),
            'val_score': val.get('validation_score'),
            'val_issues': '; '.join(val.get('issues', []))[:300],
            'val_feedback': (val.get('feedback') or '')[:300],
            'final_track': final.get('standardization_track'),
            'iterations': state.get('current_iteration', 0),
        })
    except Exception as e:
        print(f'  실패: {e}')

log_df = pd.DataFrame(log_rows)
log_path = OUTPUT_DIR / 'llm_review_log.csv'
log_df.to_csv(log_path, index=False, encoding='utf-8-sig')
print(f'\n저장: {log_path}')

## 2단계: LLM 리뷰 결과 분석

In [ ]:
print(f'=== 리뷰 결과 ({len(log_df)}건) ===\n')
print(f"val_target 분포:")
print(log_df['val_target'].value_counts())

rule_pass = log_df[log_df['val_target'] == 'rule']
llm_path = log_df[log_df['val_target'] == 'llm']

print(f'\nRule 승인: {len(rule_pass)}건 ({len(rule_pass)/len(log_df)*100:.1f}%)')
print(f"  평균 val_score: {rule_pass['val_score'].mean():.3f}")
print(f"  평균 drift: {rule_pass['drift'].mean()*100:.2f}%")

print(f'\nLLM 경로: {len(llm_path)}건 ({len(llm_path)/len(log_df)*100:.1f}%)')
print(f"  평균 val_score: {llm_path['val_score'].mean():.3f}")
print(f"  평균 drift: {llm_path['drift'].mean()*100:.2f}%")

## 3단계: Rule 승인 케이스의 LLM feedback

In [ ]:
for _, r in rule_pass.iterrows():
    print(f"[{r['version']}, {r['certification_level']}] {r['pdf'][:55]}")
    print(f"  score={r['val_score']}, drift={r['drift']:.3f}")
    print(f"  feedback: {r['val_feedback'][:200]}")
    print()

## 4단계: Option A 데이터셋 생성

V1의 Rule 기반 점수(460건) + LLM 리뷰 메타데이터(75건) 병합

In [ ]:
feat_v1 = pd.read_parquet(ROOT / 'data' / 'processed' / 'project_features.parquet')
print(f'V1 Rule 기반: {len(feat_v1)}건')

# LLM 리뷰 컬럼 이름 재정리
log_renamed = log_df.rename(columns={
    'val_target': 'llm_review_target',
    'val_is_valid': 'llm_review_is_valid',
    'val_score': 'llm_review_score',
    'val_issues': 'llm_review_issues',
    'val_feedback': 'llm_review_feedback',
    'iterations': 'llm_review_iterations',
})

log_slim = log_renamed[['project_id', 'llm_review_target', 'llm_review_is_valid',
                         'llm_review_score', 'llm_review_issues', 'llm_review_feedback',
                         'llm_review_iterations']].copy()
log_slim['project_id'] = log_slim['project_id'].astype(str)

# V1에 LLM 리뷰 붙이기
feat_v1['project_id'] = feat_v1['project_id'].astype(str)
merged = feat_v1.merge(log_slim, on='project_id', how='left')
merged['has_llm_review'] = merged['llm_review_target'].notna()

# 저장
out = ROOT / 'data' / 'processed' / 'project_features_option_a.parquet'
merged.to_parquet(out, index=False)

print(f'\n저장: {out}')
print(f'행수: {len(merged)}')
print(f'LLM 리뷰됨: {merged["has_llm_review"].sum()}')
print(f'리뷰 없음: {(~merged["has_llm_review"]).sum()}')

## 5단계: `credit_rule_hit_rate` 가 LLM 판단의 proxy인지 확인

In [ ]:
reviewed = merged[merged['has_llm_review']]

# LLM 승인 vs 거부 그룹의 credit_hit 비교
approved = reviewed[reviewed['llm_review_target'] == 'rule']
rejected = reviewed[reviewed['llm_review_target'] == 'llm']

print('credit_rule_hit_rate 평균:')
print(f'  LLM 승인 그룹: {approved["credit_rule_hit_rate"].mean():.3f}')
print(f'  LLM 거부 그룹: {rejected["credit_rule_hit_rate"].mean():.3f}')
print(f'  차이: {(approved["credit_rule_hit_rate"].mean() - rejected["credit_rule_hit_rate"].mean()):.3f}')
print('\n→ credit_hit 높은 건물일수록 LLM도 승인')

## 다음 단계

`03_SHAP_분석.ipynb`에서 XGBoost + SHAP 분석 진행.